# Step 1: ResNet-20 FP32 → INT8 golden model

Runtime → Change runtime type → **GPU**. Checkpoints go to Google Drive, so a disconnect loses nothing:
re-running the training cell resumes from the last epoch.

Design decisions: `docs/step1_decisions.md`.

In [ ]:
import os
if not os.path.exists('/content/reliable-reram-crossbar'):
    !git clone https://github.com/Shivanshmittal023/reliable-reram-crossbar.git /content/reliable-reram-crossbar
%cd /content/reliable-reram-crossbar
!git pull -q
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CK = '/content/drive/MyDrive/reram/checkpoints'
os.makedirs(CK, exist_ok=True)

Unit tests (CPU, a few seconds). Everything below assumes these pass.

In [ ]:
!python -m pytest -q

## 1a. FP32 training (200 epochs, ~1 h on a T4). Target ≥ 91.5%.

In [ ]:
!python -m scripts.train_fp32 --arch resnet20 --seed 0 --amp --out {CK}/resnet20_fp32_seed0.pt

## 1b. Fold BN → calibrate → QAT (30 epochs) → INT8 golden export

In [ ]:
!python -m scripts.calibrate_and_qat --fp32 {CK}/resnet20_fp32_seed0.pt --out-dir {CK}

## 1c. Re-check the golden model from the exported file

In [ ]:
!python -m scripts.eval_int8 --export {CK}/resnet20_int8_seed0.pt

## 1d. Commit the golden export

Copy `resnet20_int8_seed0.pt` / `.npz` into `checkpoints/` and the new row of
`results/baseline_accuracy.csv`, then commit from a laptop (D14). Download them with:

In [ ]:
from google.colab import files
for f in ['resnet20_int8_seed0.pt', 'resnet20_int8_seed0.npz']:
    files.download(f'{CK}/{f}')
files.download('results/baseline_accuracy.csv')